<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/06_29_full_fixed1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


In [2]:
import os
import shutil
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.regularizers import l2
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input


In [3]:
BASE            = "/content/newdata"
IMG_SRC         = "/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT_DIR  = "/drive/MyDrive/checkpoints"
MODEL_SAVE_PATH = (
    "/drive/MyDrive/Colab Notebooks/Models/"
    "dermoscopy/efficientnetv2s_dual_branch_v2.keras"
)

In [4]:
BATCH_SIZE    = 16
IMAGE_SIZE    = 256
FUSION_LAYER  = "block4c_add"
EPOCHS        = 40
WARMUP_EPOCHS = 3
LR_MAX        = 1e-4
LR_MIN        = 1e-6

In [5]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.20),
    layers.RandomBrightness(0.15),
], name="augmentation")

In [6]:
def mixup_batch(images, labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    lam = tf.random.uniform([], minval=0.0, maxval=1.0)
    lam = tf.maximum(lam, 1.0 - lam)
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, indices)
    mixed_labels = lam * labels + (1.0 - lam) * tf.gather(labels, indices)
    return mixed_images, mixed_labels

In [7]:
def add_edge_map(image, label):
    image = tf.cast(image, tf.float32)
    gray  = tf.image.rgb_to_grayscale(image)
    sobel = tf.image.sobel_edges(gray)
    edge  = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1))
    edge  = edge / (tf.reduce_max(edge) + 1e-6)
    rgb   = preprocess_input(image)
    return (rgb, edge), label

def prepare_dataset(path, shuffle, apply_mixup=False):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=shuffle,
    )
    ds = ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)
    if apply_mixup:
        def mixup_map(inputs, labels):
            rgb, edge = inputs
            mixed_rgb, mixed_labels = mixup_batch(rgb, labels, alpha=0.2)
            return (mixed_rgb, edge), mixed_labels
        ds = ds.map(mixup_map, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = prepare_dataset(f"{BASE}/train", shuffle=True,  apply_mixup=True)
val_ds   = prepare_dataset(f"{BASE}/valid", shuffle=False, apply_mixup=False)
test_ds  = prepare_dataset(f"{BASE}/test",  shuffle=False, apply_mixup=False)


Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [8]:
def get_class_weights(train_path):
    class_dirs = sorted([
        d for d in os.listdir(train_path)
        if os.path.isdir(os.path.join(train_path, d))
    ])
    counts  = [len(os.listdir(os.path.join(train_path, d))) for d in class_dirs]
    total   = sum(counts)
    weights = {i: total / (len(counts) * c) for i, c in enumerate(counts)}
    print("Class dirs   :", class_dirs)
    print("Class counts :", counts)
    print("Class weights:", weights)
    return weights

class_weights = get_class_weights(f"{BASE}/train")


Class dirs   : ['melanoma', 'non_melanoma']
Class counts : [890, 7122]
Class weights: {0: 4.501123595505618, 1: 0.562482448750351}


In [9]:
loss_fn = tf.keras.losses.CategoricalFocalCrossentropy(
    gamma=2.0,
    alpha=0.25,
    label_smoothing=0.05,
)

In [10]:
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, warmup_steps, total_steps, lr_max, lr_min):
        super().__init__()
        self.warmup_steps = float(warmup_steps)
        self.total_steps  = float(total_steps)
        self.lr_max = lr_max
        self.lr_min = lr_min

    def __call__(self, step):
        step  = tf.cast(step, tf.float32)
        # --- warmup phase ---
        warmup_lr = self.lr_max * (step / self.warmup_steps)
        # --- cosine decay phase ---
        decay_steps = self.total_steps - self.warmup_steps
        cos_step    = tf.minimum(step - self.warmup_steps, decay_steps)
        cos_lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (
            1.0 + tf.cos(np.pi * cos_step / decay_steps)
        )
        return tf.where(step < self.warmup_steps, warmup_lr, cos_lr)

    def get_config(self):
        return dict(
            warmup_steps=self.warmup_steps,
            total_steps=self.total_steps,
            lr_max=self.lr_max,
            lr_min=self.lr_min,
        )

steps_per_epoch = len(train_ds)
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = steps_per_epoch * WARMUP_EPOCHS

lr_schedule = WarmupCosineDecay(
    warmup_steps=warmup_steps,
    total_steps=total_steps,
    lr_max=LR_MAX,
    lr_min=LR_MIN,
)

In [11]:
def create_dual_model():
    rgb_input  = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="rgb_input")
    edge_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1), name="edge_input")

    # ── RGB BRANCH ──────────────────────────────────────────
    x_rgb = data_augmentation(rgb_input)
    x_rgb = preprocess_input(x_rgb)

    base_model = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    )
    for layer in base_model.layers[:-160]:
        layer.trainable = False
    for layer in base_model.layers[-160:]:
        layer.trainable = True

    feature_extractor = tf.keras.Model(
        inputs=base_model.input,
        outputs=base_model.get_layer(FUSION_LAYER).output,
        name="rgb_backbone",
    )
    middle_feature = feature_extractor(x_rgb)   # (B, H', W', C')

    # ── EDGE BRANCH ─────────────────────────────────────────
    x = layers.Conv2D(32,  3, activation="relu", padding="same")(edge_input)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(64,  3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)

    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.BatchNormalization()(x)

    x = layers.Resizing(middle_feature.shape[1], middle_feature.shape[2])(x)
    x = layers.Conv2D(middle_feature.shape[-1], 1, padding="same")(x)

    # ── FUSION ──────────────────────────────────────────────
    fused = layers.Concatenate()([middle_feature, x])
    fused = layers.Conv2D(256, 3, activation="relu", padding="same",
                          kernel_regularizer=l2(1e-5))(fused)
    fused = layers.BatchNormalization()(fused)

    # ── CBAM-STYLE CHANNEL ATTENTION ────────────────────────
    gap = layers.GlobalAveragePooling2D()(fused)
    gmp = layers.GlobalMaxPooling2D()(fused)

    att_gap = layers.Dense(64,  activation="relu")(gap)
    att_gap = layers.Dense(256, activation="sigmoid")(att_gap)

    att_gmp = layers.Dense(64,  activation="relu")(gmp)
    att_gmp = layers.Dense(256, activation="sigmoid")(att_gmp)

    att = layers.Add()([att_gap, att_gmp])

    fused_gap      = layers.GlobalAveragePooling2D()(fused)
    fused_weighted = layers.Multiply()([fused_gap, att])

    # ── CLASSIFIER ──────────────────────────────────────────
    x = layers.Dense(256, activation="relu", kernel_regularizer=l2(1e-5))(fused_weighted)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu", kernel_regularizer=l2(1e-5))(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(2, activation="softmax")(x)

    model = tf.keras.Model(
        inputs=[rgb_input, edge_input],
        outputs=outputs,
        name="dual_branch_v2",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        loss=loss_fn,
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return model

model = create_dual_model()
model.summary()

Model: "dual_branch_v2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ edge_input          │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │        320 │ edge_input[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256, 256,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rgb_input           │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ augmentation        │ (None, 256, 256,  │          0 │ rgb_input[0][0]   │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing (Resizing) │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rgb_backbone        │ (None, 16, 16,    │  1,317,880 │ augmentation[0][… │
│ (Functional)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 16, 16,    │     16,512 │ resizing[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 16,    │          0 │ rgb_backbone[0][… │
│ (Concatenate)       │ 256)              │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │    590,080 │ concatenate[0][0] │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │      1,024 │ conv2d_4[0][0]  

 Total params: 2,184,186 (8.33 MB)

 Trainable params: 865,346 (3.30 MB)

 Non-trainable params: 1,318,840 (5.03 MB)

In [12]:
checkpoint_best = ModelCheckpoint(
    filepath=f"{CHECKPOINT_DIR}/best_dual_{FUSION_LAYER}_v2.keras",
    monitor="val_auc",
    save_best_only=True,
    mode="max",
    verbose=1,
)

early_stop = EarlyStopping(
    monitor="val_auc",
    patience=12,          # generous: cosine LR causes occasional dips
    restore_best_weights=True,
    mode="max",
    verbose=1,
)

In [13]:
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    class_weight=class_weights,
    callbacks=[checkpoint_best, early_stop],
)

Epoch 1/40
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step - accuracy: 0.5241 - auc: 0.5312 - loss: 0.0574 - precision: 0.6132 - recall: 0.5175
Epoch 1: val_auc improved from None to 0.78754, saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add_v2.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add_v2.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 177s 303ms/step - accuracy: 0.6106 - auc: 0.6493 - loss: 0.0538 - precision: 0.6930 - recall: 0.5826 - val_accuracy: 0.6743 - val_auc: 0.7875 - val_loss: 0.0427 - val_precision: 0.6743 - val_recall: 0.6743
Epoch 2/40
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step - accuracy: 0.7440 - auc: 0.8152 - loss: 0.0481 - precision: 0.8180 - recall: 0.6898
Epoch 2: val_auc did not improve from 0.78754
501/501 ━━━━━━━━━━━━━━━━━━━━ 146s 290ms/step - accuracy: 0.7478 - auc: 0.8196 - loss: 0.0473 - precision: 0.8180 - recall: 0.6877 - val_accuracy: 0.5874 - val_auc: 0.7267 - val_loss: 0.0491 - val_precision: 0.5874 - val_recall

In [14]:
def tta_predict(model, dataset, n_aug=8):
    all_preds  = []
    true_labels = None

    for _ in range(n_aug):
        preds  = []
        labels = []
        for (rgb, edge), lbl in dataset:
            rgb_aug = data_augmentation(rgb, training=True)
            pred    = model([rgb_aug, edge], training=False)
            preds.append(pred.numpy())
            labels.append(lbl.numpy())
        all_preds.append(np.concatenate(preds, axis=0))
        if true_labels is None:
            true_labels = np.concatenate(labels, axis=0)

    return np.mean(all_preds, axis=0), true_labels

print("\nRunning TTA on test set (8-pass)...")
tta_preds, true_labels = tta_predict(model, test_ds, n_aug=8)
tta_accuracy = np.mean(
    np.argmax(tta_preds, axis=1) == np.argmax(true_labels, axis=1)
)


Running TTA on test set (8-pass)...


In [15]:
results = model.evaluate(test_ds, return_dict=True)

print("\n" + "=" * 40)
print("FINAL RESULTS")
print("=" * 40)
print(f"Fusion layer   : {FUSION_LAYER}")
print(f"Test accuracy  : {results['accuracy']:.4f}")
print(f"Test AUC       : {results['auc']:.4f}")
print(f"Test precision : {results['precision']:.4f}")
print(f"Test recall    : {results['recall']:.4f}")
print(f"TTA accuracy   : {tta_accuracy:.4f}  (8-pass average)")


63/63 ━━━━━━━━━━━━━━━━━━━━ 15s 232ms/step - accuracy: 0.7555 - auc: 0.8825 - loss: 0.0342 - precision: 0.7555 - recall: 0.7555

FINAL RESULTS
Fusion layer   : block4c_add
Test accuracy  : 0.7555
Test AUC       : 0.8825
Test precision : 0.7555
Test recall    : 0.7555
TTA accuracy   : 0.8723  (8-pass average)


In [16]:
model.save(MODEL_SAVE_PATH)
print(f"\nModel saved to {MODEL_SAVE_PATH}")


Model saved to /drive/MyDrive/Colab Notebooks/Models/dermoscopy/efficientnetv2s_dual_branch_v2.keras
